In [1]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"📍 Using device: {device}")

📍 Using device: cuda


In [2]:
import os
from PIL import Image, UnidentifiedImageError
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm import tqdm


def filter_valid_images(image_paths, labels=None):
    valid_paths = []
    valid_labels = []
    for i, path in enumerate(tqdm(image_paths, desc="🔍 유효 이미지 필터링")):
        try:
            img = Image.open(path).convert("RGB")
            img.verify()  # 이미지 유효성 검증
            valid_paths.append(path)
            if labels is not None:
                valid_labels.append(labels[i])
        except (UnidentifiedImageError, OSError):
            continue  # 깨진 이미지 스킵
    return valid_paths, valid_labels if labels is not None else valid_paths


# 📦 학습용 전처리 + 증강 조합
transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.1),
    transforms.RandomRotation(degrees=8),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# 🧪 검증/테스트용 전처리 (증강 없음!)
transform_test = transforms.Compose([
    transforms.Resize((224, 224)),                  # 이미지 크기 통일
    transforms.ToTensor(),                          # PIL → 텐서
    transforms.Normalize(mean=[0.485, 0.456, 0.406],# 정규화 (학습과 동일)
                         std=[0.229, 0.224, 0.225])
])

class CarDataset(Dataset):
    def __init__(self, image_paths, labels=None, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        try:
            image = Image.open(img_path).convert("RGB")
        except (UnidentifiedImageError, OSError):
            print(f"⚠️ 이미지 오류: {img_path}")
            return self.__getitem__((idx + 1) % len(self.image_paths))  # 다음 이미지로 대체

        if self.transform:
            image = self.transform(image)

        if self.labels is not None:
            return image, self.labels[idx]
        else:
            return image

In [3]:
from PIL import Image, UnidentifiedImageError
from tqdm import tqdm

def filter_valid_images(image_paths, labels=None):
    valid_paths = []
    valid_labels = []
    for i, path in enumerate(tqdm(image_paths, desc="🔍 유효 이미지 필터링")):
        try:
            img = Image.open(path).convert("RGB")
            img.verify()  # 이미지가 완전한지 검사
            valid_paths.append(path)
            if labels is not None:
                valid_labels.append(labels[i])
        except (UnidentifiedImageError, OSError):
            continue  # 깨진 이미지 스킵
    return valid_paths, valid_labels if labels is not None else valid_paths


In [4]:
train_dir = "/kaggle/input/hecto-ai/train"
class_names = sorted(os.listdir(train_dir))  # 총 396개 클래스
class_to_idx = {cls_name: idx for idx, cls_name in enumerate(class_names)}

image_paths = []
labels = []

for cls_name in class_names:
    cls_folder = os.path.join(train_dir, cls_name)
    for img_file in os.listdir(cls_folder):
        image_paths.append(os.path.join(cls_folder, img_file))
        labels.append(class_to_idx[cls_name])

In [5]:
image_paths, labels = filter_valid_images(image_paths, labels)

🔍 유효 이미지 필터링: 100%|██████████| 33137/33137 [05:41<00:00, 97.08it/s] 


In [6]:
from sklearn.model_selection import train_test_split

train_paths, val_paths, train_labels, val_labels = train_test_split(
    image_paths, labels, test_size=0.1, stratify=labels, random_state=42
)

In [7]:
from torchvision import transforms

transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.1),
    transforms.RandomRotation(degrees=8),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

transform_test = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])


In [8]:
from torch.utils.data import Dataset

class CarDataset(Dataset):
    def __init__(self, image_paths, labels=None, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        try:
            image = Image.open(img_path).convert("RGB")
        except (UnidentifiedImageError, OSError):
            print(f"⚠️ 오류 이미지: {img_path}")
            return self.__getitem__((idx + 1) % len(self.image_paths))  # 다음 이미지 대체

        if self.transform:
            image = self.transform(image)

        if self.labels is not None:
            return image, self.labels[idx]
        else:
            return image


In [9]:
from torch.utils.data import DataLoader

train_dataset = CarDataset(train_paths, train_labels, transform=transform_train)
val_dataset = CarDataset(val_paths, val_labels, transform=transform_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=2)

In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from tqdm import tqdm

# 🔧 모델 정의 (ResNet18, 출력 396개)
model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 396)
model = model.to(device)

# 🎯 손실 함수 및 옵티마이저
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=3e-4)

# 🔁 Warm Restarts Scheduler
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=5, T_mult=1)

# 🔄 학습 루프
EPOCHS = 30
best_val_acc = 0

for epoch in range(EPOCHS):
    print(f"\n🌀 Epoch {epoch+1}/{EPOCHS}")

    # 🔹 학습
    model.train()
    train_loss = 0.0
    correct = 0
    total = 0

    train_loop = tqdm(train_loader, desc="🟦 Training", leave=False)
    for images, labels in train_loop:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        scheduler.step()  # ⬅️ 여기에 warm restart 적용

        train_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        train_loop.set_postfix(loss=loss.item(), acc=100 * correct / total)

    epoch_train_loss = train_loss / len(train_loader)
    epoch_train_acc = correct / total

    # 🔹 검증
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    val_loop = tqdm(val_loader, desc="🟪 Validating", leave=False)
    with torch.no_grad():
        for images, labels in val_loop:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

            val_loop.set_postfix(loss=loss.item(), acc=100 * val_correct / val_total)

    epoch_val_loss = val_loss / len(val_loader)
    epoch_val_acc = val_correct / val_total

    # 📢 에폭 결과 출력
    print(f"📘 Train Loss: {epoch_train_loss:.4f}, Train Acc: {epoch_train_acc:.4f}")
    print(f"📗 Val   Loss: {epoch_val_loss:.4f}, Val   Acc: {epoch_val_acc:.4f}")

    # 💾 최고 모델 저장
    if epoch_val_acc > best_val_acc:
        best_val_acc = epoch_val_acc
        torch.save(model.state_dict(), "best_model_396_res18_warmrestart.pth")
        print("✅ Best model saved!")



🌀 Epoch 1/30


📘 Train Loss: 5.5550, Train Acc: 0.0274
📗 Val   Loss: 4.7251, Val   Acc: 0.0794
✅ Best model saved!

🌀 Epoch 2/30


📘 Train Loss: 4.0084, Train Acc: 0.1843
📗 Val   Loss: 3.3537, Val   Acc: 0.2333
✅ Best model saved!

🌀 Epoch 3/30


📘 Train Loss: 2.5555, Train Acc: 0.4369
📗 Val   Loss: 2.4107, Val   Acc: 0.4185
✅ Best model saved!

🌀 Epoch 4/30


📘 Train Loss: 1.5811, Train Acc: 0.6360
📗 Val   Loss: 1.5337, Val   Acc: 0.6086
✅ Best model saved!

🌀 Epoch 5/30


...

📘 Train Loss: 0.0676, Train Acc: 0.9812
📗 Val   Loss: 0.5152, Val   Acc: 0.8672

🌀 Epoch 27/30


📘 Train Loss: 0.0754, Train Acc: 0.9783
📗 Val   Loss: 0.5084, Val   Acc: 0.8672

🌀 Epoch 28/30


📘 Train Loss: 0.0822, Train Acc: 0.9772
📗 Val   Loss: 0.5006, Val   Acc: 0.8712

🌀 Epoch 29/30


📘 Train Loss: 0.0649, Train Acc: 0.9810
📗 Val   Loss: 0.4370, Val   Acc: 0.8893
✅ Best model saved!

🌀 Epoch 30/30


📘 Train Loss: 0.0595, Train Acc: 0.9829
📗 Val   Loss: 0.4985, Val   Acc: 0.8799


In [12]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import torchvision.transforms as transforms

# ✅ 환경 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
test_root = "/kaggle/input/hecto-ai/test"
model_path = "./best_model_396_res18_warmrestart.pth"

# ✅ 전처리 정의 (학습 시와 동일해야 함)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# ✅ 데이터셋 정의
class TestDataset(Dataset):
    def __init__(self, dataframe, root_dir, transform=None):
        self.dataframe = dataframe
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_path = os.path.join(self.root_dir, self.dataframe.iloc[idx]['img_path'])
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image

# ✅ 데이터 불러오기
test_df = pd.read_csv("/kaggle/input/hecto-ai/test.csv")
test_df['img_path'] = test_df['img_path'].str.replace("test/", "", regex=False)
sample_sub = pd.read_csv("/kaggle/input/hecto-ai/sample_submission.csv")

# ✅ class 순서 맞추기
class_order = sample_sub.columns[1:]  # 'ID' 제외한 클래스 열들
class_to_index = {cls_name: i for i, cls_name in enumerate(class_order)}

# ✅ 모델 불러오기
model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, len(class_order))  # 클래스 수 = 396
model.load_state_dict(torch.load(model_path, map_location=device))
model.to(device)
model.eval()

# ✅ 테스트셋 준비
test_dataset = TestDataset(test_df, test_root, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# ✅ 추론 및 확률 저장
all_probs = []

with torch.no_grad():
    for images in tqdm(test_loader, desc="🔍 Inference"):
        images = images.to(device)
        outputs = model(images)
        probs = F.softmax(outputs, dim=1).cpu().numpy()
        all_probs.extend(probs)

# ✅ DataFrame으로 변환 (클래스 순서에 맞게)
submission = pd.DataFrame(all_probs, columns=class_order)
submission.insert(0, "ID", test_df["ID"])  # ID 컬럼 맨 앞에 삽입

# ✅ 저장
submission.to_csv("submission.csv", index=False)
print("✅ 'submission.csv' 저장 완료 🎯")


🔍 Inference: 100%|██████████| 130/130 [02:24<00:00,  1.11s/it]


✅ 'submission.csv' 저장 완료 🎯
